In [ ]:
!pip install openai langchain langchain-openai SPARQLWrapper pandas requests

In [ ]:
import os
import pandas as pd
print(os.getcwd())
#%cd content
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from SPARQLWrapper import SPARQLWrapper, JSON
import requests
import random
import io
from getpass import getpass
import chardet
from urllib.parse import urlparse, quote, unquote
import re
import json
import subprocess

In [ ]:
repo_url = "https://github.com/OpenData-IQ/open-data-benchmark"
repo_dir = "open-data-benchmark"

# Fetch all folders at first level
top_level_items = os.listdir('.')
print(top_level_items)

# Check if this exact folder exists at the top level
if repo_dir not in top_level_items:
    # Folder does not exist → Clone
    !git clone {repo_url}
else:
    # Folder exists → Execute pull
    !cd {repo_dir} && git pull

In [ ]:
from pathlib import Path
import shutil

folder_path = Path("questions")

# If the folder exists, remove all its contents
if folder_path.exists() and folder_path.is_dir():
    # Iterate and remove each item in the folder
    for item in folder_path.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()
    print(f"Contents of '{folder_path}' deleted.")
else:
    folder_path.mkdir(parents=True, exist_ok=True)
    print(f"Folder '{folder_path}' created.")

# Ensure the folder exists at the end
folder_path.mkdir(parents=True, exist_ok=True)

# Read the dataframe
source_df = pd.read_csv(f"{repo_dir}/cleaned_questions_dataset.csv", encoding="utf-8")

In [ ]:
# Replaces German umlauts according to CKAN/GovData convention
def replace_umlauts(text):
    replacements = {
        'ä': 'ae', 'ö': 'oe', 'ü': 'ue',
        'Ä': 'Ae', 'Ö': 'Oe', 'Ü': 'Ue',
        'ß': 'ss'
    }
    for umlaut, replacement in replacements.items():
        text = text.replace(umlaut, replacement)
    return text


GOV_DATA_BASE = "https://www.govdata.de/ckan/dataset/"

# Step 1: Filter the DataFrame
filter = "simple"
task_type = 1
comment = ""
filtered_df = source_df[(source_df["frage_typ"] == filter) & (source_df["datengrundlage"] == grundlage)]
len(filtered_df)
# Step 2: Select exactly 5 random rows (or fewer if not enough available)
n_rows = 5
sampled_rows = filtered_df.sample(n=min(n_rows, len(filtered_df)))
print(sampled_rows)
system_message = SystemMessage(content="""You are a professional creator of an Open Data benchmark.
Your task is to formulate correct question-answer pairs for a given file (distribution of a dataset).
Keep in mind that the benchmark should resemble real questions from end users, i.e., do not explicitly
refer to a file in your question formulation. Provide the answer in JSON format.
{
  "question": "string",
  "answer": "string"
}
""")
OPENAI_API_KEY=getpass('Enter your OpenAI API key: ')
llm = ChatOpenAI(
    model="gpt-4.1-2025-04-14",
    temperature=0.0,
    api_key=OPENAI_API_KEY
    #max_tokens=max_tok,
)
examples = []
# Step 3: Process each selected row
for idx, row in sampled_rows.iterrows():
  question = row["question"]
  answer = row["answer"]
  examples.append({
      "question": question,
      "answer": answer
    }
  )
prompt_template = FewShotPromptTemplate(
            examples=examples,
            example_prompt=PromptTemplate.from_template(
                "### Example ###\n"
                "Question: {question}\n"
                "Answer: {answer}\n"
            ),
            suffix="""### Now create a question-answer pair based on the dataset\n
            Title: {title}\n
            Description: {description}\n
            As well as the following data (in German)\n
            {input} ###
            """,
            input_variables=["title", "description", "input"],
)

processable_csv = False
while not processable_csv:
  rand_int = random.randint(1, 18287)
  print(rand_int)
  sparql = SPARQLWrapper("https://www.govdata.de/sparql")
  query = f"""
    PREFIX dcat: <http://www.w3.org/ns/dcat#>
    PREFIX dct: <http://purl.org/dc/terms/>
    SELECT ?dataset ?title ?description ?csvDist_title ?csvDist_description ?downloadURL WHERE {{
    {{
      SELECT ?dataset ?title ?csvDist ?csvDist_title ?csvDist_description ?downloadURL WHERE {{
        ?dataset a dcat:Dataset ;
             dcat:distribution ?csvDist ;
             dct:title ?title .
        ?csvDist dct:format ?format ;
             dct:title ?csvDist_title ;
             dct:description ?csvDist_description;
             dcat:downloadURL ?downloadURL .
        FILTER (
          LCASE(STR(?format)) = "csv" ||
          LCASE(STR(?format)) = "text/csv" ||
          ?format = <http://publications.europa.eu/resource/authority/file-type/csv> ||
          ?format = <http://publications.europa.eu/resource/authority/file-type/CSV>
        )
      }}
      ORDER BY ?dataset ?csvDist
    }}
  }}
  OFFSET {rand_int}
  LIMIT 1
    """
  sparql.setQuery(query)
  sparql.setReturnFormat(JSON)
  results = sparql.query().convert()
  bindings = results["results"]["bindings"]

  if bindings:
        download_url = bindings[0]["downloadURL"]["value"]
        dataset = bindings[0]["dataset"]["value"]
        title = bindings[0]["title"]["value"]
        csv_description = bindings[0]["csvDist_description"]["value"]
        csv_title = bindings[0]["csvDist_title"]["value"]
        print("Found URL:", download_url)

        try:
            response = requests.get(download_url)
            if response.status_code == 200:
                parsed_url = urlparse(download_url)
                real_file_name = os.path.basename(parsed_url.path)
                temp_file_name = "download"
                with open(folder_path / temp_file_name, "wb") as f:
                    f.write(response.content)
                print("File downloaded successfully.")
                # Check whether the files ends with .csv
                if download_url.lower().endswith(".csv"):
                    print("File has .csv ending. Ready to process.")
                    file_name = None
                    cd = response.headers.get("Content-Disposition")
                    if cd:
                      match = re.findall('filename="?([^"]+)"?', cd)
                      if match:
                        file_name = match[0].strip()

                    # In case there is no filename in header, try fetching it from URL
                    if not file_name:
                      parsed_url = urlparse(download_url)
                      file_name = os.path.basename(parsed_url.path)
                      if not file_name:
                        # If no filename is present in the URL, use the title (URL-encoded) with a .csv extension.
                        file_name = quote(csv_title.strip().replace(" ", "_")) + ".csv"

                    # Rename temporary file in final file
                    os.rename(folder_path / temp_file_name, folder_path / file_name)


                    with open(folder_path / file_name, "rb") as f:
                      result = chardet.detect(f.read(10000))  # read first 10KB
                      encoding = result['encoding']
                      print(f"Detected encoding: {encoding}")
                    # Load with detected encoding
                    df = pd.read_csv(folder_path / file_name, encoding=encoding)
                    processable_csv = True
                    prompt_string = prompt_template.invoke({
                      "title": csv_title,
                      "description": csv_description,
                      #"input": str(df.head(10))
                      "input": str(df),
                    }).to_string()
                    print(prompt_string)
                    messages = [
                      system_message,
                      HumanMessage(content=prompt_string)
                    ]
                    generated = llm.invoke(messages).content
                    json_generated = json.loads(generated)
                    question = json_generated["question"]
                    answer = json_generated["answer"]
                    # Extract the identifier part from the URL
                    metadata_file = ""
                    parsed = urlparse(dataset)
                    identifier_encoded = parsed.path.rstrip("/").split("/")[-1]
                    # Decode the URL-encoded part (z.B. %C3%BC → ü)
                    last_part = unquote(identifier_encoded)
                    # First try via Govdata without umlauts
                    metadata_name = replace_umlauts(last_part)
                    metadata_file = f"{metadata_name}.rdf"
                    govdata_url = GOV_DATA_BASE + metadata_file
                    govmeta_response = requests.get(govdata_url)
                    if govmeta_response.status_code == 200:
                      with open(folder_path / metadata_file, "wb") as govmeta_f:
                          govmeta_f.write(govmeta_response.content)
                          print("Metadata downloaded successfully")
                    else:
                      print(f"Problem with Govdata metadata download. Searching metadata decription in the source portal")
                      source_url = dataset+".rdf"
                      print(source_url)
                      metadata_name = last_part
                      # Try to fetch the RDF metadata description from the source data portal
                      sourcemeta_response = requests.get(source_url)
                      if sourcemeta_response.status_code == 200:
                        metadata_file = f"{metadata_name}.rdf"
                        with open(folder_path / metadata_file, "wb") as sourcemeta_f:
                          sourcemeta_f.write(sourcemeta_response.content)
                          print(f"Metadata downloaded. Please check whether it is actually RDF. Otherwise search '{title}' in Govdata.")
                      else:
                          print(f"Metadata download not possible. Search '{title}' in Govdata.")
                else:
                    print("File does not have .csv ending. Retrying...\n")
            else:
                print(f"Failed to download: HTTP {response.status_code}")
        except Exception as e:
            print("Error during download:", e)
  else:
        print("No result found. Retrying...\n")

In [ ]:
print(f"Question: {question}")
print(f"Answer: {answer}")
print(f"Task Type: {task_type}")
print(f"Question Type: {filter}")
print(f"File name: {file_name}")
print(f"Metadata: {metadata_file}")
print(f"Remark: {comment}")

In [ ]:
# Define the new row as a dictionary
new_row = {
    "question": question,
    "antwort": answer,
    "task_type": task_type,
    "question_type": filter,
    "filename": file_name,
    "metadata": metadata_file,
    "remark": comment

}
# Append the new row
source_df = pd.concat([source_df, pd.DataFrame([new_row])], ignore_index=True)

# Copy the files
source_df.to_csv(f"{repo_dir}/cleaned_questions_dataset.csv", index=False)

shutil.copy(
    folder_path / file_name,
    Path(f"{repo_dir}/daten") / file_name
)

shutil.copy(
    folder_path / metadata_file,
    Path(f"{repo_dir}/metadaten") / metadata_file
)

In [ ]:
TOKEN = ""
# Change working directory
%cd {repo_dir}

# Configure Git user identity (required for commits)
subprocess.run(["git", "config", "user.email", ""], check=True)
subprocess.run(["git", "config", "user.name", ""], check=True)

result = subprocess.run(["git", "status"], capture_output=True, text=True)
print(result.stdout)

# Stage everything
subprocess.run(["git", "add", "."], check=True)

# Commit
subprocess.run(["git", "commit", "-m", "Update via Colab"], check=True)

# Push using the same token
subprocess.run(["git", "push", f"https://{TOKEN}@github.com/OpenData-IQ/open-data-benchmark"], check=True)

print("Successfully pushed to repository")

# Go back to parent directory
os.chdir("..")

# Confirm you're in the right place
print("Current directory:", os.getcwd())